In [1]:
import numpy as np 
import pandas as pd 


In [2]:
df = pd.read_csv('social_media_sentiment_test.csv')
df.head()

,text,label
0,omg it was stnadard 🫤,Neutral
1,highkeyyyyy this is perfect 😁,Positive
2,omg it was standard 🙂,Neutral
3,ngl thisssss is terrible 💔,Negative
4,highkey this is lit ✨,Positive


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2000 non-null   object
 1   label   2000 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


In [4]:
df.duplicated().sum()

np.int64(0)

In [6]:
unique_label = df['label'].unique()
label_numbers = {}
i = 0
for lab in unique_label:
    label_numbers[lab] = i 
    i+=1

df['label'] = df['label'].map(label_numbers)

In [8]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [9]:
import string 
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [10]:
df['text'].apply(remove_punc)

0               omg it was stnadard 🫤
1       highkeyyyyy this is perfect 😁
2               omg it was standard 🙂
3          ngl thisssss is terrible 💔
4               highkey this is lit ✨
                    ...              
1995      bro thisssss is amazinggg 🔥
1996              ngl this is trash 😒
1997              lol this is awful 😞
1998      highkeyyy it was okayyyyy 😐
1999          bro this is fantatsic 😍
Name: text, Length: 2000, dtype: object

In [11]:
def remove_numbers(txt):
    new=""
    for i in txt: 
        if not i.isdigit(): 
            new = new+i
    return new 
df['text'] = df['text'].apply(remove_numbers)

In [12]:
import nltk 
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [13]:
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [14]:
def remove(txt):
    words = txt.split()
    cleaned = [ ]
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [16]:
df['text'] = df['text'].apply(remove)

In [17]:
df.loc[1]['text']

'highkeyyyyy perfect 😁'

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['text'], df['label'],test_size=0.20, random_state=42)

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy

for model, score in results.items():
    print(model, ":", score)

Naive Bayes : 0.965
Logistic Regression : 0.9625
SVM : 0.9725
Random Forest : 0.9475


In [22]:
import pandas as pd

comparison = pd.DataFrame(results.items(), columns=["Model","Accuracy"])
print(comparison)

                 Model  Accuracy
0          Naive Bayes    0.9650
1  Logistic Regression    0.9625
2                  SVM    0.9725
3        Random Forest    0.9475


In [23]:
best_model = max(results, key=results.get)
print("Best Model:", best_model)

Best Model: SVM


In [25]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)

model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)

pickle.dump(model, open("Reaction_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))